In [4]:
from pathlib import Path
import h5py

# Change this to the .h5 file you want to inspect
h5_path = Path(r"../data/h5_features")
if h5_path.is_dir():
    candidates = sorted(h5_path.glob("*.h5"))
    if not candidates:
        raise FileNotFoundError(f"No .h5 files found in {h5_path}")
    h5_path = candidates[1]

print(f"Using file: {h5_path}")
with h5py.File(h5_path, "r") as f:
    features = f["features"]
    coords = f["coords"]
    print("features shape:", features.shape)
    print("coords shape:", coords.shape)
    if features.ndim >= 2:
        print("feature dimension:", features.shape[1])
    else:
        print("feature dimension: unknown")

Using file: ..\data\h5_features\TCGA-5T-A9QA-01Z-00-DX1.B4212117-E0A7-4EF2-B324-8396042ACEC1.h5
features shape: (61902, 1536)
coords shape: (61902, 2)
feature dimension: 1536


In [3]:
from pathlib import Path
import pandas as pd

# Update these paths to your two manifest/text files
file_a = Path(r"../data/reference/manifests/batch_0001.txt")
file_b = Path(r"../data/reference/temp/batch_0001.txt")

def load_filenames(path: Path) -> set[str]:
    df = pd.read_csv(path, sep="\t")
    if "filename" not in df.columns:
        raise ValueError(f"filename column not found in {path}")
    return set(df["filename"].dropna().astype(str).str.strip())

names_a = load_filenames(file_a)
names_b = load_filenames(file_b)

common = names_a & names_b
only_a = names_a - names_b
only_b = names_b - names_a

print(f"File A: {file_a}")
print(f"File B: {file_b}")
print(f"Total A: {len(names_a)} | Total B: {len(names_b)}")
print(f"Common: {len(common)} | Only A: {len(only_a)} | Only B: {len(only_b)}")

# Show a small sample
print("Sample common:", sorted(list(common))[:5])
print("Sample only A:", sorted(list(only_a))[:5])
print("Sample only B:", sorted(list(only_b))[:5])

File A: ..\data\reference\manifests\batch_0001.txt
File B: ..\data\reference\temp\batch_0001.txt
Total A: 19 | Total B: 47
Common: 19 | Only A: 0 | Only B: 28
Sample common: ['TCGA-A2-A0EV-01Z-00-DX1.EA8C5594-BA4F-47A8-949B-D536E00E62C9.svs', 'TCGA-AC-A6NO-01Z-00-DX1.61B7F48C-6D6E-4C1B-B236-DD130ECBDA9D.svs', 'TCGA-AC-A8OS-01Z-00-DX1.3FD44846-8BD2-4A7A-9A87-8D3D29C25F60.svs', 'TCGA-AN-A0FJ-01Z-00-DX1.97B60767-916E-4938-9D0B-E6C0FE1CB3FC.svs', 'TCGA-AN-A0XO-01Z-00-DX1.204E554E-B1A8-41AD-8E39-62484DF4E3CD.svs']
Sample only A: []
Sample only B: ['TCGA-A2-A0EV-01A-01-BSA.895f4272-acc0-48e5-8c91-58209e6aa849.svs', 'TCGA-A2-A0EV-01A-01-TSA.49658afd-55bd-4adb-a3c2-06a4e633efc5.svs', 'TCGA-AC-A6NO-01A-01-TS1.EC103F22-AE26-43CA-983C-E7CD7AA9F633.svs', 'TCGA-AC-A8OS-01A-01-TS1.31C88D4F-7621-4B24-83CC-8FD69EC72933.svs', 'TCGA-AN-A0FJ-01A-01-BSA.cef21074-da54-4b5d-859f-1ad06b1b7c7e.svs']


In [7]:
import pandas as pd
import mygene
import requests
import gseapy as gp
from collections import defaultdict

# Cấu hình
TCGA_FILE = "../data/csv/Python_Input_Genomic_LogCPM.csv" 
 
# 1. Đọc dữ liệu 
import time # Nhớ import thư viện time ở đầu file nhé

# 1. Đọc dữ liệu
print("\n" + "="*50)
print("Đang nạp ma trận khổng lồ (Fat Matrix) vào RAM...")  

start_time = time.time()

# Dùng bộ máy mặc định, tắt low_memory để nhồi thẳng toàn bộ vào RAM
df = pd.read_csv(TCGA_FILE, index_col=0, low_memory=False)

end_time = time.time()
print(f"\n-> THÀNH CÔNG! Đã nạp xong trong {round(end_time - start_time, 1)} giây.")
print(f"Kích thước dữ liệu: {df.shape[0]} bệnh nhân, {df.shape[1]} gene.")

# 2. Xử lý Ensembl ID version
ens_to_full = {}
for full_id in df.columns:
    base = full_id.split('.')[0]
    ens_to_full[full_id] = base
base_ids = list(set(ens_to_full.values()))
print(f"Số Ensembl ID duy nhất (bỏ version): {len(base_ids)}")

# 3. Chuyển đổi sang Symbol
mg = mygene.MyGeneInfo()
print("Đang query mygene (có thể mất vài phút)...")
results = mg.querymany(base_ids, scopes='ensembl.gene', fields='symbol', species='human', returnall=True)
base_to_sym = {}
for item in results['out']:
    if 'symbol' in item and item['symbol']:
        base_to_sym[item['query']] = item['symbol']
print(f"Đã ánh xạ {len(base_to_sym)}/{len(base_ids)} base Ensembl ID thành công.")

# Gán symbol cho các cột có version
full_to_sym = {}
for full, base in ens_to_full.items():
    if base in base_to_sym:
        full_to_sym[full] = base_to_sym[base]

mapped_columns = list(full_to_sym.keys())
df_mapped = df[mapped_columns].copy()
df_mapped.columns = [full_to_sym[col] for col in mapped_columns]

# Gộp các cột cùng symbol
print("Gộp các Ensembl ID cùng symbol (tính trung bình)...")
df_symbol = df_mapped.T.groupby(level=0).mean().T
print(f"Ma trận cuối cùng với Symbol: {df_symbol.shape[1]} gene.")

# 4. Lấy Hallmark gene sets từ MSigDB bằng gseapy
print("\nĐang kết nối với MSigDB qua gseapy...")
msig = gp.Msigdb()  # tạo instance Msigdb

# Lấy các bộ Hallmark (thử dbver="2026.1.Hs", nếu lỗi sẽ tự động chọn bản mới nhất)
try:
    hallmark_sets = msig.get_gmt(category='h.all', dbver="2026.1.Hs")
    print("Dùng dbver=2026.1.Hs")
except Exception:
    # fallback về phiên bản mới nhất có sẵn
    hallmark_sets = msig.get_gmt(category='h.all')
    print("Dùng bản mới nhất có sẵn")

# hallmark_sets là dict dạng {pathway_name: [gene_symbol1, gene_symbol2, ...]}
print(f"Đã tải {len(hallmark_sets)} bộ Hallmark.")

# 5. Phân loại thành 6 nhóm theo từ khóa
def classify_pathway(name):
    n = name.upper()
    if any(k in n for k in ['E2F', 'G2M', 'MITOTIC']):
        return 'CellCycle'
    if any(k in n for k in ['DNA_REPAIR', 'P53', 'UV_RESPONSE']):
        return 'DNADamage'
    if any(k in n for k in ['EPITHELIAL_MESENCHYMAL', 'EMT', 'ANGIOGENESIS', 'APICAL', 'HEDGEHOG']):
        return 'EMT'
    if any(k in n for k in ['ESTROGEN', 'ANDROGEN']):
        return 'Hormone'
    if any(k in n for k in ['INFLAMMATORY', 'INTERFERON', 'TNFA', 'IL6', 'JAK',
                             'ALLOGRAFT', 'COMPLEMENT', 'COAGULATION']):
        return 'Immune'
    return 'Other'

group_genes = defaultdict(set)
for pw, genes in hallmark_sets.items():
    group_genes[classify_pathway(pw)].update(genes)

print("Các nhóm và số gene (trước lọc):")
for grp, genes in group_genes.items():
    print(f"  {grp}: {len(genes)} gene")

# 6. Tạo bag cho từng nhóm
bag_data = {}
for grp, symbols in group_genes.items():
    available = [s for s in symbols if s in df_symbol.columns]
    if not available:
        print(f"Bag '{grp}': không có gene nào trong dữ liệu bệnh nhân.")
        continue
    bag_df = df_symbol[available].reset_index()
    bag_df.rename(columns={bag_df.columns[0]: 'submitter_id'}, inplace=True)
    bag_data[grp] = bag_df
    print(f"Bag '{grp}': {bag_df.shape[1]-1} gene, {bag_df.shape[0]} bệnh nhân")

# 7. Lưu file CSV
for grp, bag in bag_data.items():
    fname = f"../data/csv/bag_{grp}.csv"
    bag.to_csv(fname, index=False)
    print(f"Đã lưu {fname}")

print("Hoàn tất!")






Đang nạp ma trận khổng lồ (Fat Matrix) vào RAM...

-> THÀNH CÔNG! Đã nạp xong trong 15.8 giây.
Kích thước dữ liệu: 2267 bệnh nhân, 60660 gene.
Số Ensembl ID duy nhất (bỏ version): 60616
Đang query mygene (có thể mất vài phút)...

-> THÀNH CÔNG! Đã nạp xong trong 15.8 giây.
Kích thước dữ liệu: 2267 bệnh nhân, 60660 gene.
Số Ensembl ID duy nhất (bỏ version): 60616
Đang query mygene (có thể mất vài phút)...


29 input query terms found dup hits:	[('ENSG00000235059', 2), ('ENSG00000287478', 2), ('ENSG00000226506', 2), ('ENSG00000280018', 2), ('E
1259 input query terms found no hit:	['ENSG00000233517', 'ENSG00000257918', 'ENSG00000224167', 'ENSG00000272917', 'ENSG00000131484', 'ENS
1259 input query terms found no hit:	['ENSG00000233517', 'ENSG00000257918', 'ENSG00000224167', 'ENSG00000272917', 'ENSG00000131484', 'ENS


Đã ánh xạ 45045/60616 base Ensembl ID thành công.
Gộp các Ensembl ID cùng symbol (tính trung bình)...
Gộp các Ensembl ID cùng symbol (tính trung bình)...
Ma trận cuối cùng với Symbol: 43998 gene.

Đang kết nối với MSigDB qua gseapy...
Ma trận cuối cùng với Symbol: 43998 gene.

Đang kết nối với MSigDB qua gseapy...
Dùng dbver=2026.1.Hs
Đã tải 50 bộ Hallmark.
Các nhóm và số gene (trước lọc):
  Other: 2751 gene
  Immune: 906 gene
  Hormone: 388 gene
  EMT: 470 gene
  DNADamage: 625 gene
  CellCycle: 484 gene
Bag 'Other': 2746 gene, 2267 bệnh nhân
Bag 'Immune': 905 gene, 2267 bệnh nhân
Bag 'Hormone': 388 gene, 2267 bệnh nhân
Bag 'EMT': 470 gene, 2267 bệnh nhân
Bag 'DNADamage': 624 gene, 2267 bệnh nhân
Bag 'CellCycle': 482 gene, 2267 bệnh nhân
Dùng dbver=2026.1.Hs
Đã tải 50 bộ Hallmark.
Các nhóm và số gene (trước lọc):
  Other: 2751 gene
  Immune: 906 gene
  Hormone: 388 gene
  EMT: 470 gene
  DNADamage: 625 gene
  CellCycle: 484 gene
Bag 'Other': 2746 gene, 2267 bệnh nhân
Bag 'Immune': 905

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import pickle
from pathlib import Path

# --- LỚP MÔ HÌNH GENENCODER ---
class GenEncoder(nn.Module):
    def __init__(self, bag_info):
        super(GenEncoder, self).__init__()
        self.encoders = nn.ModuleDict()
        for name, num_genes in bag_info.items():
            self.encoders[name] = nn.Sequential(
                nn.BatchNorm1d(num_genes), 
                nn.Linear(num_genes, 512),
                nn.ReLU(),
                nn.BatchNorm1d(512)
            )

    def forward(self, x_dict):
        output_vectors = {}
        for name, data in x_dict.items():
            output_vectors[name] = self.encoders[name](data)
        return output_vectors

# --- CẤU HÌNH ---
data_dir = Path("../data/csv")
bag_files = {
    'CellCycle': data_dir / 'bag_CellCycle.csv',
    'DNADamage': data_dir / 'bag_DNADamage.csv',
    'EMT': data_dir / 'bag_EMT.csv',
    'Hormone': data_dir / 'bag_Hormone.csv',
    'Immune': data_dir / 'bag_Immune.csv',
    'Other': data_dir / 'bag_Other.csv'
}

bag_info = {}
data_dict = {}

# --- BƯỚC 1: ĐỌC DỮ LIỆU ---
submitter_ids = None
for name, file in bag_files.items():
    df = pd.read_csv(file)
    if submitter_ids is None:
        submitter_ids = df['submitter_id']
    
    num_genes = df.shape[1] - 1
    bag_info[name] = num_genes
    data_dict[name] = torch.tensor(df.drop(columns=['submitter_id']).values, dtype=torch.float32)

# --- BƯỚC 2: KHỞI TẠO VÀ CHẠY DỮ LIỆU QUA MODEL ---
model = GenEncoder(bag_info)
model.eval() # Chế độ trích xuất đặc trưng

with torch.no_grad():
    # ĐÂY LÀ BƯỚC BẠN THIẾU: Phải chạy dữ liệu qua model
    encoded_outputs = model(data_dict)

# --- BƯỚC 3: XUẤT 6 FILE RIÊNG BIỆT ---
print("Đang xuất 6 file riêng biệt...")
data_dir.mkdir(parents=True, exist_ok=True)
for name, vec in encoded_outputs.items():
    encoded_df = pd.DataFrame(vec.numpy(), columns=[f'feat_{i}' for i in range(512)])
    encoded_df.insert(0, 'submitter_id', submitter_ids)
    output_path = data_dir / f"encoded_{name}.csv"
    encoded_df.to_csv(output_path, index=False)
    print(f"Đã lưu: {output_path}")

# --- BƯỚC 4: STACK THÀNH [N or 6, 512] ---
# Dùng thứ tự cố định để đảm bảo 6 túi luôn ở vị trí giống nhau
ordered_keys = ['CellCycle', 'DNADamage', 'EMT', 'Hormone', 'Immune', 'Other']
tensors_list = [encoded_outputs[k] for k in ordered_keys]

combined_matrix = torch.stack(tensors_list, dim=1)

print(f"\nShape cuối cùng của ma trận: {combined_matrix.shape}")

# --- BƯỚC 5: LƯU MA TRẬN 3D ---
combined_path = data_dir / "combined_genomic_features.pkl"
with open(combined_path, "wb") as f:
    pickle.dump(combined_matrix, f)

print(f"Đã lưu ma trận [N, 6, 512] vào '{combined_path}'")


Đang xuất 6 file riêng biệt...
Đã lưu: ..\data\csv\encoded_CellCycle.csv
Đã lưu: ..\data\csv\encoded_CellCycle.csv
Đã lưu: ..\data\csv\encoded_DNADamage.csv
Đã lưu: ..\data\csv\encoded_DNADamage.csv
Đã lưu: ..\data\csv\encoded_EMT.csv
Đã lưu: ..\data\csv\encoded_EMT.csv
Đã lưu: ..\data\csv\encoded_Hormone.csv
Đã lưu: ..\data\csv\encoded_Hormone.csv
Đã lưu: ..\data\csv\encoded_Immune.csv
Đã lưu: ..\data\csv\encoded_Immune.csv
Đã lưu: ..\data\csv\encoded_Other.csv

Shape cuối cùng của ma trận: torch.Size([2267, 6, 512])
Đã lưu ma trận [N, 6, 512] vào '..\data\csv\combined_genomic_features.pkl'
Đã lưu: ..\data\csv\encoded_Other.csv

Shape cuối cùng của ma trận: torch.Size([2267, 6, 512])
Đã lưu ma trận [N, 6, 512] vào '..\data\csv\combined_genomic_features.pkl'


In [ ]:
import torch
from co_attention import GenomicGuidedCoAttention

# Example shapes: Q (6, 512), K/V (N, 512)
q = torch.randn(6, 512)
k = torch.randn(1000, 512)
v = torch.randn(1000, 512)

model = GenomicGuidedCoAttention(dim=512)
out, attn = model(q, k, v)

print("Output shape:", out.shape)   # (6, 512)
print("Attn shape:", attn.shape)     # (6, 1000)

# Batched example: Q (B, 6, 512), K/V (B, N, 512)
q_b = torch.randn(2, 6, 512)
k_b = torch.randn(2, 1000, 512)
v_b = torch.randn(2, 1000, 512)

out_b, attn_b = model(q_b, k_b, v_b)
print("Batched output shape:", out_b.shape)  # (2, 6, 512)
print("Batched attn shape:", attn_b.shape)    # (2, 6, 1000)

ModuleNotFoundError: No module named 'feature_fusion_network'

In [5]:
from pathlib import Path

import numpy as np, torch, h5py
 
slide_id='TCGA-E9-A5FL-01Z-00-DX1'[:24]
h5_dir=Path('data/h5_features')
attn_dir=Path('data/outputs/co_attention')
h5_matches=sorted(h5_dir.glob(f'{slide_id}*.h5'))
print('H5 match count:', len(h5_matches))
h5_path=h5_matches
f=h5py.File(h5_path,'r')
coords=f['coords'][()]
f.close()
attn_path=None
 
for pattern in (f'{slide_id}*_co_attention.pt', f'{slide_id}*.npy'):
    matches=sorted(attn_dir.glob(pattern))
    
    if matches:
        attn_path=matches[0]
        
        break

print('Attention path:', attn_path)
 
if attn_path is None: raise SystemExit('No attention file found')

if attn_path.suffix.lower()=='.pt':
    payload=torch.load(attn_path, map_location='cpu')
    
    weights=payload['attention'] if isinstance(payload, dict) else payload
    
    weights=weights.detach().cpu().numpy() if isinstance(weights, torch.Tensor) else np.asarray(weights)
else:
    weights=np.load(attn_path)
weights=weights.squeeze()

weights_1d=weights.mean(axis=(0,1)) if weights.ndim==3 else (weights.mean(axis=0) if weights.ndim==2 else weights)

print('H5:', h5_path)
print('coords length:', coords.shape[0])
print('attention raw shape:', weights.shape)
print('attention 1D length:', weights_1d.shape[0])

H5 match count: 0


TypeError: expected str, bytes or os.PathLike object, not list

In [6]:

print(h5_path)

[]
